# Exploratory Data Analysis — Xente / BNPL transaction data

This notebook explores the partner transaction feed used for the Bati Bank credit-risk proxy model. Place the Kaggle **Xente Challenge** CSV at `data/raw/Xente_challenge_dataset.csv` (see README).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAW = Path("data/raw/Xente_challenge_dataset.csv")
if not RAW.exists():
    raise FileNotFoundError(
        f"Expected dataset at {RAW}. Download from Kaggle and copy the CSV into data/raw/."
    )

df = pd.read_csv(RAW)
df["TransactionStartTime"] = pd.to_datetime(df["TransactionStartTime"], errors="coerce")
print("Shape:", df.shape)
df.head()

## 1. Structure and dtypes
Row/column counts, dtypes, and cardinality.

In [ ]:
print(df.info())
print("\nUnique customers:", df["CustomerId"].nunique())
print("Date range:", df["TransactionStartTime"].min(), "→", df["TransactionStartTime"].max())

## 2. Summary statistics
Central tendency and spread for numeric fields.

In [ ]:
df[["Amount", "Value", "FraudResult"]].describe()

## 3. Distributions (numeric)
Skewness and heavy tails motivate log transforms and robust scaling in `data_processing.py`.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
np.log1p(df["Value"].clip(lower=0)).hist(bins=40, ax=ax[0])
ax[0].set_title("log1p(Value)")
df["FraudResult"].value_counts(normalize=True).plot(kind="bar", ax=ax[1], title="FraudResult share")
plt.tight_layout()

## 4. Categorical distributions
Channel and product mix inform dominant-category features.

In [ ]:
for col in ["ProductCategory", "ChannelId"]:
    if col in df.columns:
        print(col)
        print(df[col].value_counts().head(15))

## 5. Correlations
Linear associations among numeric columns.

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[num_cols].corr(numeric_only=True)
corr

## 6. Missing values
Guides imputation vs drop decisions in the pipeline.

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
missing[missing > 0]

## 7. Outliers (boxplot)
Extreme `Value`/`Amount` tails suggest robust aggregates and fraud-aware rates.

In [ ]:
plt.figure(figsize=(6, 4))
plt.boxplot(np.log1p(df["Value"].clip(lower=0)), vert=False)
plt.title("log1p(Value) boxplot")
plt.show()

## Top insights (3–5)

1. **Heavy-tailed spend**: Transaction `Value` is right-skewed; customer-level aggregates and `log1p` on RFM components stabilize clustering for the proxy target.
2. **Low fraud prevalence**: `FraudResult` is typically imbalanced; we expose `fraud_rate` per customer as a weak risk signal, not a default label.
3. **Channel and category concentration**: A few `ProductCategory` / `ChannelId` levels dominate; dominant-category encoding plus WoE reduces dimensionality versus full one-hot on sparse IDs.
4. **Temporal coverage**: The calendar span of `TransactionStartTime` sets the RFM snapshot; drift in partner seasonality is a monitoring risk for any proxy model.
5. **No repayment label**: All supervised signal is behavioral; model metrics measure separation of *engagement segments*, not literal default until bank labels exist.
